# 16.2 Unions, Narrowing, `Literal` and `TypedDict`

**Prerequisites:** 16.1 The Type System, 2.5 Dictionaries, 5.3 Dataclasses and Enums  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- `X | Y` unions, and why `Optional` is the one you meet most
- **Narrowing** — how a checker proves a union is safe to use
- 🔴 The truthiness trap: `if value:` is *not* `if value is not None:`
- `Literal` — types made of specific **values**
- 🔴 **Exhaustiveness with `assert_never`** — the checker tells you when you add a state
- `Final` and `ClassVar` — declaring what must not change
- `TypedDict` in depth: `NotRequired`, `Required`, and `total=False`
- Choosing between `TypedDict`, `dataclass` and `NamedTuple`
- `Annotated` — attaching metadata to a type

---

## Real data is full of unions

Almost every interesting value in a real program is "one of several things":

- a lookup that may miss — `str | None`
- a job state — `"queued" | "running" | "done"`
- an ID that arrives as either — `int | str`
- a result or an error

**16.1** showed the checker catching `None` misuse. This notebook is about expressing those
shapes precisely, and about **narrowing** — the reasoning a checker does to prove that by the
time you use a value, it is definitely the one you think.

```
   str | None                     narrowing                    str
   ┌──────────────┐         if value is None: return      ┌──────────┐
   │ "eu" or None │  ────────────────────────────────>    │   "eu"   │
   └──────────────┘                                       └──────────┘
   cannot .upper()                                        .upper() is safe
```

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py162_"))


def mypy(name, source=None, *flags):
    """Type-check a file with mypy and return its report."""
    if source is not None:
        (WORK / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    done = subprocess.run(
        [sys.executable, "-m", "mypy", name,
         "--cache-dir", str(WORK / ".mypy_cache"),
         "--no-color-output", "--no-error-summary", *flags],
        cwd=WORK, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=300)
    report = (done.stdout + done.stderr).strip() or "(mypy found nothing to report)"
    return (f"$ mypy {name} {' '.join(flags)}".rstrip() + "\n" + "-" * 68 + "\n"
            + report + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


def write(name, source):
    """Write a script into the scratch directory and return its name."""
    (WORK / name).write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return name


def python(name):
    done = subprocess.run([sys.executable, name], cwd=WORK, capture_output=True,
                          text=True, encoding="utf-8", errors="replace", timeout=60)
    return (f"$ python {name}\n" + "-" * 68 + "\n"
            + (done.stdout + done.stderr).strip() + "\n" + "-" * 68
            + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## `X | Y` and `Optional`

Since **3.10** the union operator works in annotations:

```python
def find(uid: int) -> str | None: ...        # modern
def find(uid: int) -> Optional[str]: ...     # older; identical meaning
def find(uid: int) -> Union[str, None]: ...  # older still
```

🔴 **`Optional[X]` means `X | None`. It does not mean "optional argument".** A parameter with a
default is optional; a parameter typed `Optional[str]` *must still be passed*, it may just be
`None`. That confusion is worth naming because it is extremely common.

```python
def f(a: str | None)            -> a is required, may be None
def f(a: str = "eu")            -> a is optional, never None
def f(a: str | None = None)     -> a is optional AND may be None
```

In [ ]:
print(mypy("optional.py", r"""
    def find_region(uid: int) -> str | None:
        return {1: "eu", 2: "us"}.get(uid)


    def label_broken(uid: int) -> str:
        region = find_region(uid)
        return region.upper()                # 🔴 region may be None


    def label_narrowed(uid: int) -> str:
        region = find_region(uid)
        if region is None:
            return "UNKNOWN"
        reveal_type(region)                  # narrowed by the early return
        return region.upper()


    def label_asserted(uid: int) -> str:
        region = find_region(uid)
        assert region is not None, "caller guarantees a known uid"
        reveal_type(region)
        return region.upper()
"""))

Three functions, one error. `label_narrowed` and `label_asserted` both end up
with `region` revealed as **`str`** — the checker followed the control flow.

### The narrowing forms

| You write | The checker learns |
|---|---|
| `if x is None: return` | after it, `x` is not `None` |
| `if isinstance(x, str):` | inside, `x` is `str` |
| `if not isinstance(x, str): raise` | after it, `x` is `str` |
| `assert x is not None` | after it, `x` is not `None` |
| `if x == "queued":` | inside, `x` is `Literal["queued"]` — for `Literal` types |
| `match x: case str():` | per branch (**3.4**) |
| `if isinstance(x, str) and x:` | both facts |

🔴 `assert` narrows — but remember **15.1**: `python -O` deletes `assert` statements. The
*type* narrowing survives (it happened at check time), but the runtime guarantee does not. Use
`assert` for invariants, `if ... raise` for anything that must hold in production.

## 🔴 The truthiness trap

`if value:` and `if value is not None:` narrow differently, and the difference is a real bug
source — empty strings, `0`, and empty lists are all falsy but **not** `None`.

In [ ]:
print(python(write("truthy.py", r"""
    def timeout_for(raw: int | None) -> int:
        # 🔴 BUG: 0 is a legitimate timeout, but it is also falsy
        if not raw:
            return 30
        return raw


    def timeout_for_correct(raw: int | None) -> int:
        if raw is None:
            return 30
        return raw


    for value in (5, 0, None):
        print(f"  raw={str(value):5} truthy-check={timeout_for(value):3} "
              f"is-None-check={timeout_for_correct(value):3}")
""")))
print()
print("🔴 raw=0 asked for NO timeout and silently got 30.")
print("   mypy cannot catch this - both versions are perfectly typed.")

Both functions type-check cleanly. Only one is correct.

This is **16.1**'s lesson again in miniature: the types are right, the logic is wrong. Reach
for `is None` whenever `None` is meaningfully different from "empty" or "zero" — which is
almost always.

## `Literal` — types made of values

`Literal` lets a type be a specific set of **values**, not just a class. It is how you type a
job state, a mode flag, or an HTTP method without inventing a class.

In [ ]:
print(mypy("literal.py", r"""
    from typing import Literal

    State = Literal["queued", "running", "done", "failed"]
    Mode = Literal["r", "w", "a"]


    def advance(state: State) -> State:
        return "running"


    def open_spool(path: str, mode: Mode = "r") -> str:
        return f"{path}:{mode}"


    good: State = "queued"
    bad: State = "cancelled"                 # 🔴 not one of the four

    open_spool("jobs.log", "w")
    open_spool("jobs.log", "x")              # 🔴 not a valid mode

    def describe(state: State) -> str:
        if state == "queued":
            reveal_type(state)               # narrowed to one literal
        return str(state)
"""))

`"cancelled"` and `"x"` were both rejected, and inside the `==` branch the
checker narrowed `state` to `Literal['queued']`.

> **`Literal` vs `Enum` (5.3).** Both model a closed set. `Literal` is lighter and works
> naturally with JSON and external data, where the value really *is* the string. `Enum` gives
> you a namespace, methods and `auto()`, and prevents typos at runtime rather than only at check
> time. Use `Literal` at the boundary, `Enum` inside your domain.

## 🔴 Exhaustiveness — `assert_never`

This is the single most valuable pattern in this notebook.

You have a function handling every state. Six months later someone adds a fifth state. **How do
you find every place that needs updating?** `assert_never` makes the checker do it: it accepts
only a value of type `Never`, so it type-checks *only if* the checker has narrowed the union to
nothing.

In [ ]:
print(mypy("exhaustive.py", r"""
    from typing import Literal, assert_never

    State = Literal["queued", "running", "done", "failed"]


    def describe_complete(state: State) -> str:
        if state == "queued":
            return "waiting to start"
        elif state == "running":
            return "in progress"
        elif state == "done":
            return "finished"
        elif state == "failed":
            return "gave up"
        else:
            assert_never(state)          # all four handled - no error


    def describe_incomplete(state: State) -> str:
        if state == "queued":
            return "waiting to start"
        elif state == "running":
            return "in progress"
        else:
            assert_never(state)          # 🔴 'done' and 'failed' unhandled
"""))

The error names exactly what is unhandled:
`expected "Never"` with the leftover literals. Add a fifth state to `State` and **every**
`assert_never` in the codebase reports it — a compile-time list of everything you must update.

The same works with `Enum` members and with `match`/`case` (**3.4**):

```python
match state:
    case "queued":  ...
    case "running": ...
    case _ as unreachable:
        assert_never(unreachable)
```

🔴 Without `assert_never`, the `else` branch just returns something plausible and the new state
silently takes the wrong path. This pattern converts a class of silent bug into a checker error.

## `Final` and `ClassVar`

| Declaration | Means |
|---|---|
| `x: Final = 30` | never reassigned |
| `x: Final[int] = 30` | the same, with an explicit type |
| `x: ClassVar[int] = 0` | belongs to the **class**, not instances (**5.1**) |

`Final` is a *checker* guarantee, not a runtime one — nothing stops reassignment at runtime,
exactly as in **16.1**.

In [ ]:
print(mypy("final.py", r"""
    from typing import ClassVar, Final

    MAX_ATTEMPTS: Final = 3
    CEILING: Final[float] = 30.0

    MAX_ATTEMPTS = 5                        # 🔴 cannot reassign a Final


    class JobRunner:
        registry: ClassVar[dict[str, int]] = {}      # shared by all instances
        limit: Final[int]

        def __init__(self, limit: int) -> None:
            self.limit = limit

        def bump(self) -> None:
            self.limit += 1                 # 🔴 Final attribute


    reveal_type(MAX_ATTEMPTS)               # Literal[3], because Final narrows it
"""))

Note the last line: `MAX_ATTEMPTS: Final = 3` is revealed as **`Literal[3]?`**,
not `int`. `Final` tells the checker the value can never change, so it keeps the *exact* value —
which then works with `Literal` types and narrowing.

## `TypedDict` in depth

**5.3** introduced `TypedDict` for JSON-shaped data. Here is the rest of it: which keys are
required, and what the checker actually enforces.

In [ ]:
print(mypy("tdict.py", r"""
    from typing import NotRequired, Required, TypedDict


    class JobSpec(TypedDict):
        job_id: str
        attempts: int
        region: NotRequired[str]            # may be absent


    class PartialSpec(TypedDict, total=False):
        job_id: Required[str]               # required despite total=False
        attempts: int                       # optional
        region: str                         # optional


    spec: JobSpec = {"job_id": "build-1", "attempts": 0}
    spec["region"] = "eu"
    reveal_type(spec["attempts"])

    missing: JobSpec = {"job_id": "build-1"}          # 🔴 attempts is required
    wrong: JobSpec = {"job_id": "b", "attempts": "3"} # 🔴 wrong value type
    spec["nonsense"] = 1                              # 🔴 unknown key

    partial: PartialSpec = {"job_id": "build-2"}      # fine - the rest optional
"""))

Four distinct errors, each naming the exact problem. Two things worth
knowing:

- 🔴 **A `TypedDict` is a plain `dict` at runtime.** No validation, no methods, no `__init__`.
  It is purely a checker-level description, so data arriving from JSON that does not match
  produces **no error at all** until something touches a missing key.
- `total=False` makes every key optional; `Required[...]` and `NotRequired[...]` (3.11) let you
  mix, which is almost always clearer than two classes.

### Choosing between the three

| | `TypedDict` | `dataclass` (**5.3**) | `NamedTuple` |
|---|---|---|---|
| Runtime type | `dict` | a real class | a `tuple` |
| Access | `spec["job_id"]` | `spec.job_id` | `spec.job_id` or `spec[0]` |
| Mutable | yes | yes (unless `frozen=True`) | **no** |
| Methods / validation | ❌ | ✅ `__post_init__` | limited |
| Defaults | ❌ | ✅ | ✅ |
| Cost to create | zero — it *is* the dict | an object | a tuple |
| Best for | **JSON in and out** | 🔴 **your domain model** | fixed records, tuple unpacking |

The rule that serves best: **`TypedDict` at the boundary, `dataclass` inside.** Parse the
incoming JSON as a `TypedDict`, convert it into a `dataclass` once, and let the rest of the
program work with real objects that can carry behaviour.

In [ ]:
print(python(write("shapes.py", r"""
    from dataclasses import dataclass
    from typing import NamedTuple, TypedDict


    class SpecDict(TypedDict):
        job_id: str
        attempts: int


    @dataclass
    class SpecClass:
        job_id: str
        attempts: int = 0

        @property
        def can_retry(self) -> bool:
            return self.attempts < 3


    class SpecTuple(NamedTuple):
        job_id: str
        attempts: int = 0


    as_dict = SpecDict(job_id="build-1", attempts=1)
    as_class = SpecClass("build-1", 1)
    as_tuple = SpecTuple("build-1", 1)

    print("TypedDict  :", type(as_dict).__name__, "|", as_dict, "| access:", as_dict["job_id"])
    print("dataclass  :", type(as_class).__name__, "|", as_class, "| can_retry:", as_class.can_retry)
    print("NamedTuple :", type(as_tuple).__name__, "|", as_tuple, "| unpacks:", tuple(as_tuple))

    job_id, attempts = as_tuple
    print()
    print("  the NamedTuple unpacked to:", job_id, attempts)
    print("  🔴 note TypedDict's runtime type is plain 'dict' - the class vanished")
""")))

`type(as_dict).__name__` is **`dict`** — the `SpecDict` class does not exist at
runtime in any meaningful sense. The `dataclass` carries a property; the `NamedTuple` unpacks.

## `Annotated` — metadata on a type

`Annotated[T, ...]` attaches arbitrary extra information to a type. The checker sees only `T`
and ignores the rest; *other tools* read the metadata.

This is the mechanism behind `pydantic`'s validators and FastAPI's dependency injection
(**18**), and it is how you attach units, constraints or documentation without inventing a new
type.

In [ ]:
print(mypy("annotated.py", r"""
    from typing import Annotated, get_type_hints


    class Range:
        def __init__(self, low: int, high: int) -> None:
            self.low, self.high = low, high

        def __repr__(self) -> str:
            return f"Range({self.low}, {self.high})"


    Attempts = Annotated[int, Range(0, 10)]
    Seconds = Annotated[float, "unit: seconds"]


    def schedule(attempts: Attempts, delay: Seconds) -> None: ...


    schedule(3, 1.5)
    schedule("three", 1.5)          # 🔴 the checker still sees `int`

    reveal_type(schedule)
"""))
print()
print(python(write("annot_rt.py", r"""
    from typing import Annotated, get_type_hints

    Seconds = Annotated[float, "unit: seconds"]

    def schedule(delay: Seconds) -> None: ...

    print("plain hints    :", get_type_hints(schedule))
    print("with metadata  :", get_type_hints(schedule, include_extras=True))
""")))

The checker enforced `int` and ignored the `Range(0, 10)` — but
`get_type_hints(..., include_extras=True)` recovered it at runtime. 🔴 By default
`get_type_hints` **strips** the metadata; you have to ask for it.

That split is the whole design: static checking and runtime metadata living in one annotation
without either interfering with the other.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **`if value:` when you mean `if value is not None:`.** `0`, `“”` and `[]` are falsy but not `None`, and both versions type-check.
2. 🔴 **Reading `Optional[X]` as “optional argument”.** It means `X | None`; the argument is still required unless it also has a default.
3. **Relying on `assert` for a production guarantee.** It narrows for the checker, but `python -O` removes it (**15.1**). Use `if ... raise`.
4. **Omitting `assert_never` in an exhaustive `if`/`match`.** Adding a state then silently takes the fallback branch instead of failing the build.
5. **Expecting `TypedDict` to validate incoming JSON.** It is a `dict` at runtime with no checking whatsoever — that is `pydantic`'s job (**18**).
6. **Using `TypedDict` for your domain model.** No methods, no defaults, no validation. Parse into a `dataclass` at the boundary.
7. **Expecting `Final` to prevent runtime reassignment.** It is a checker rule, like every other annotation.
8. **Forgetting `include_extras=True`** and wondering where your `Annotated` metadata went.
9. **Narrowing a variable and then reassigning it.** The checker widens it straight back; use a new name for the narrowed value.

## Best Practices

- Prefer `X | None` to `Optional[X]` on 3.10+ — it reads as what it is.
- Narrow with `is None`, not truthiness, unless empty and missing genuinely mean the same.
- Use `Literal` for closed sets of values crossing a boundary; `Enum` for domain concepts.
- 🔴 Put `assert_never` in the final `else` of every exhaustive branch — it is the cheapest future-proofing in the type system.
- Mark true constants `Final`; the checker then keeps their exact value.
- Use `TypedDict` at the edges and convert to a `dataclass` for anything with behaviour.
- Prefer `Required`/`NotRequired` over splitting one shape into two `TypedDict` classes.
- Reach for `Annotated` when a value needs metadata a type alone cannot carry.

## Practice Exercises

Try these before moving on.

1. 🔴 Write `timeout_for` with the truthiness bug, then a pytest test (**15.1**) that catches it. Why could mypy never have found this?
2. Type a `State` as a `Literal` of four values, write a `describe` covering three, and add `assert_never`. Read the error, then fix it. Now add a fifth state and count how many errors appear.
3. Rewrite the same thing with an `Enum` (**5.3**) and `match`/`case` (**3.4**). Which reads better, and which would you use for data arriving as JSON?
4. Define a `JobSpec` TypedDict, then feed it a dict that is missing a required key at **runtime**. What happens, and when?
5. Convert a `TypedDict` into a `dataclass` with a `from_dict` classmethod. Where should validation live?
6. Declare `MAX_ATTEMPTS: Final = 3` and `reveal_type` it. Why is it `Literal[3]` and not `int`, and where does that turn out to be useful?
7. Build `Annotated[int, Range(0, 10)]` and write a decorator (**4.4**) that reads the metadata with `get_type_hints(..., include_extras=True)` and enforces the range at runtime. You have just written a tiny `pydantic`.
8. **Interview question:** what is the difference between `Any`, `object` and `None` as a return annotation?

---

## Version notes

| Version | Change |
|---|---|
| **3.13** | `ReadOnly[...]` for `TypedDict` items; `TypeIs` — a sharper `TypeGuard` |
| **3.11** | `Required` / `NotRequired` for `TypedDict`; `assert_never`; `LiteralString` |
| **3.10** | 🔴 `X | Y` union syntax; `TypeGuard` for user-defined narrowing |
| **3.9** | `Annotated` moved into `typing` from `typing_extensions` |
| **3.8** | `Literal`, `Final`, `TypedDict` introduced (PEPs 586, 591, 589) |

## Where next

| Notebook | Covers |
|---|---|
| **16.3** | generics — `TypeVar`, PEP 695 syntax, variance, `ParamSpec` |
| **16.4** | protocols and structural typing, `Self`, `@overload` |

## Related

- **16.1** — `Any` vs `object`, and what checking cannot catch
- **2.5 Dictionaries** — `.get()` returning `None`, the most common union you will meet
- **5.3 Dataclasses and Enums** — the alternatives to `TypedDict`
- **3.4 Structural Pattern Matching** — `match`/`case`, which narrows too
- **15.1** — why `assert` is not a production guarantee
- **18 Working with APIs** — `pydantic`, which turns `Annotated` metadata into real validation